## LIBRARIES

In [ ]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as f, types
import findspark
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
import os
from pathlib import Path
import geopandas as gpd
from shapely import wkt
import json
from aux_functions_wealth import *

## Paths

In [ ]:
projectRoot = Path.cwd().parents[2]

CLEAN_DATA_YELLOW = projectRoot / "data" / "clean" / "clean_tlc_trip_record" / "yellow"
CLEAN_DATA_FHVHV = projectRoot / "data" / "clean" / "clean_tlc_trip_record" / "fhvhv"
# csv files of plot data
OUTPUT_Y_POIS_DEMAND = str(projectRoot / "data" / "clean" /"wealthAnalysis" / "yellow_zone_demand_pois.parquet")
OUTPUT_FHV_POIS_DEMAND = str(projectRoot / "data" / "clean" /"wealthAnalysis" / "fhv_zone_demand_pois.parquet")
RAW_POIS_PATH = projectRoot / "data" / "raw" / "pois"
CLEAN_POIS_PATH = projectRoot / "data" / "clean" / "pois"

## SPARK CONFIGURATION

In [ ]:
findspark.init()

#Adjust these settings based on your system's resources and requirements


# spark = (SparkSession.builder
#         .master("local[6]")  
#         .config("spark.driver.memory", "8g")
#         .config("spark.executor.memory", "8g")
#         .config("spark.sql.shuffle.partitions", "100")
#         .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64m")
#         .config("spark.driver.maxResultSize", "1g")
#         .config("spark.sql.session.timeZone", "America/New_York")
#         .config("spark.sql.timestampType", "TIMESTAMP_NTZ")
#         # Network binding fixes for macOS
#         .config("spark.driver.host", "127.0.0.1")
#         .config("spark.driver.bindAddress", "127.0.0.1")
#         .getOrCreate())

spark = (SparkSession.builder
        .master("local[*]")  
        .config("spark.driver.memory", "16g")  
        .config("spark.executor.memory", "16g") 
        .config("spark.memory.fraction", "0.8") 
        .config("spark.sql.shuffle.partitions", "200")
        .config("spark.sql.adaptive.enabled", "true") 
        .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128m") 
        .config("spark.driver.maxResultSize", "4g") 
        .config("spark.sql.session.timeZone", "America/New_York")
        .config("spark.sql.timestampType", "TIMESTAMP_NTZ")
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .getOrCreate())

## LOAD DATASETS

In [ ]:
# YELLOW
yellow = spark.read.parquet(str(CLEAN_DATA_YELLOW))
print(yellow.columns)
yellow = yellow.select(
        f.col("PULocationID").cast(types.IntegerType()),
        f.col("DOLocationID").cast(types.IntegerType()),
        f.col("pickup_datetime"),
        f.col("dropoff_datetime"),
        f.col("trip_distance").alias("trip_miles"),
        f.col("trip_time").cast("long").alias("trip_time"),
        f.col("total_amount"),
        f.col("tip_amount").alias("tips"),
        f.col("pickup_borough"),
        f.col("pickup_zone"),
        f.col("dropoff_borough"),
        f.col("dropoff_zone")
    )

# FHVHV
fhv = spark.read.parquet(str(CLEAN_DATA_FHVHV))
print(fhv.columns)
fhv = fhv.select(
        f.col("PULocationID").cast(types.IntegerType()),
        f.col("DOLocationID").cast(types.IntegerType()),
        f.col("pickup_datetime"),
        f.col("dropoff_datetime"),
        f.col("trip_miles"),
        f.col("trip_time").cast("long").alias("trip_time"),
        f.col("total_amount"),
        f.col("tips"),
        f.col("pickup_borough"),
        f.col("pickup_zone"),
        f.col("dropoff_borough"),
        f.col("dropoff_zone")
    )

In [ ]:
yellow, fhv = count_trips(yellow, fhv)
yellow, fhv = time_band_calc(yellow, fhv)
yellow, fhv = tip_pct_calc(yellow, fhv)
n_days_yellow, n_days_fhv = range_days(yellow, fhv)

## ENRICHING WEALTH INDEX THROUGH POIs


In order to enhance the previously inferred wealth index (or purchasing power distribution) across New York City, derived from taxi and Uber trip data, we incorporate an additional layer of analysis based on relevant points of interest (POIs).

This approach enables a more nuanced interpretation of mobility patterns by contextualizing them within the urban environmente. 

### Food and Dining
The spatial distribution of food-related establishments provides further socioeconomic insight. High-end restaurants, including Michelin-starred venues, are expected to correlate with higher-income areas, whereas community kitchens or more affordable dining options may indicate lower-income neighborhoods.

### Healthcare and Education
We can study the different type of hospitals and schools in the area. These includes private, governament, community and religious schools and hospitals. These locations contribute to understanding both the quality of life and the economic profile of different areas.

### Retail and Community Services
- Various retail categories, including jewelry stores, delicatessens, golf-related shops, second-hand stores, pawnbrokers, money lenders, and general variety stores
- Community-oriented facilities such as community centers and social service institutions

In [ ]:
POI_GROUPS = {
    'transport': [
        'public_transport:station'
    ],
    'food_drink': [
        'amenity:restaurant', 'amenity:kitchen'
    ],
    'healthcare': [
        'amenity:hospital'
    ],
    'entertainment': [
        'amenity:theatre', 'tourism:gallery'
    ],
    'nightlife_alcohol': [
        'amenity:nightclub', 'amenity:casino'
    ],
    'education': [
        'amenity:school'
    ],
    'commercial_expensive': [
        'shop:jewelry', 'shop:deli', 'shop:golf'
    ],
    'commercial_cheaper': [
        'shop:second_hand', 'shop:pawnbroker', 'shop:money_lender', 'shop:variety_store '
    ],
    'community': [
        'amenity:community_centre', 'amenity:social_facility'
    ]
}


In [ ]:
raw_pois = pd.read_parquet(RAW_POIS_PATH)
for i in raw_pois[raw_pois['category'] == 'hospital']['tags']:
    if i.get('operator:type') is not None:
        print(i)

In [ ]:
operadores_unicos = raw_pois[raw_pois['category'] == 'hospital']['tags'].apply(lambda x: x.get('operator')).unique()

for i in operadores_unicos:
    print(i)

In [ ]:
# SELECTION
opera = ['Metropolitan Opera House', 'New York City Center', 'BAM Howard Gilman Opera House']
ballet = ['Alvin Ailey American Dance Theater', 'Brooklyn Ballet', 'Mark Morris Dance Center']
philarmonic = ['David Geffen Hall','Stern Auditorium','Weill Recital Hall', 'Zankel Hall', 'Merkin Concert Hall',
    'Pabst Concert Hall', 'Sarah Vaughan Concert Hall']
broadway = ['Gershwin Theatre','Lyceum Theatre','Majestic Theatre','New Amsterdam Theatre','Richard Rodgers Theatre',
    'Winter Garden Theatre','Imperial Theatre','August Wilson Theatre','Minskoff Theatre','Belasco Theatre','Hudson Theatre',
    'Radio City Music Hall', 'Lunt-Fontanne Theatre','Barrymore Theatre','Bernard B. Jacobs Theatre','Samuel J. Friedman Theatre']

In [ ]:
def asign_new_category(name:str)->str | None:
    name = str(name).strip()
    if name in opera: return 'theatre_opera'
    if name in ballet: return 'theatre_ballet'
    if name in philarmonic: return 'theatre_philarmonic'
    if name in broadway: return 'theatre_broadway'
    return None

In [ ]:
df_theatre = raw_pois[raw_pois['category'] == 'theatre']
df_theatre['category'] = df_theatre['name'].apply(asign_new_category)
df_theatre = df_theatre.dropna(subset='category')

In [ ]:
#save parquets in raw pois
df_opera = df_theatre[df_theatre['category'] == 'theatre_opera']
df_ballet = df_theatre[df_theatre['category'] == 'theatre_ballet']
df_phil = df_theatre[df_theatre['category'] == 'theatre_philarmonic']
df_brod = df_theatre[df_theatre['category'] == 'theatre_broadway']

def _save_data(df: pd.DataFrame, output_path: str, output_name:str):
        data_path = projectRoot / output_path / output_name
        data_path.parent.mkdir(parents=True, exist_ok=True)

        df.to_parquet(data_path, index=False, engine='pyarrow', compression='snappy')

for df, name in zip([df_opera, df_ballet, df_phil, df_brod], 
              ['amenity_theatre_opera.parquet', 'amenity_theatre_ballet.parquet', 
               'amenity_theatre_philarmonic.parquet', 'amenity_theatre_broadway.parquet']):   
    if (len(df) != 0):
                _save_data(df, RAW_POIS_PATH, name)
    else:
        print(f"Skipping saving of data of len {len(df)} records.")

In [ ]:
import sys
if str(projectRoot) not in sys.path:
    sys.path.append(str(projectRoot))

from scripts.openstreetmap_data import clean_pois
clean_pois()

In conclusion, we have to ignore some of the POIs initially selected due to the fact that the tags do not include important information as if the hospital is public or private. This information was in the documentation, though as it was not compulsary, it was not included in the data downloaded. The same happended with the genre of the theatre, however in this case we separated them by opera, ballet, philarmonic and brodway.

In [ ]:
clean_pois_counts = pd.read_parquet(CLEAN_POIS_PATH / "zones_with_poi_counts.parquet")

In [ ]:
#the high socioeconomic pois probably are no representive enough, will take into account the origin of the trips to the areas where those places are located
high_socioeconomic_pois = {'amenity_theatre_ballet': 1,
       'amenity_theatre_broadway': 1, 'amenity_theatre_opera': 1,
       'amenity_theatre_philarmonic': 1,
       'tourism_gallery':1}

low_socioeconomic_pois = {'amenity_community_centre': -1, 'amenity_social_facility': -1, 'shop_money_lender': -3, 'shop_pawnbroker': -3,
       'shop_second_hand': 0, 'shop_variety_store ': 0}


In [ ]:
clean_pois_counts['high_SES_count'] = sum(clean_pois_counts[col]*w for col, w in high_socioeconomic_pois.items())
clean_pois_counts['low_SES_count'] = sum(clean_pois_counts[col]*w for col, w in low_socioeconomic_pois.items())
df_pois = clean_pois_counts[['id', 'zone', 'borough', 'geometry', 'length_miles', 'area_miles2','high_SES_count', 'low_SES_count']]
df_pois['high_SES_density'] = df_pois['high_SES_count'] / df_pois['area_miles2']
df_pois['low_SES_density'] = df_pois['low_SES_count'] / df_pois['area_miles2']
df_pois['SES_index'] = df_pois['high_SES_density'] - df_pois['low_SES_density']

In [ ]:
df_pois_spark = spark.createDataFrame(df_pois)
df_pois_2 = df_pois_spark.withColumn("is_high_SES", f.col("high_SES_count") > 0)
df_pois_2 = df_pois_2.select(['id','zone','borough', 'is_high_SES'])

yellow_pois = yellow.join(f.broadcast(df_pois_2), yellow.DOLocationID == df_pois_spark.id, "left")
fhv_pois = fhv.join(f.broadcast(df_pois_2), fhv.DOLocationID == df_pois_spark.id, "left")
agg_yellow_pois, agg_fhv_pois = aggregations_yellow_fhv_pois(yellow_pois, fhv_pois, n_days_yellow, n_days_fhv)
result_yellow, result_fhv = pickup_time_band_results(yellow_pois, fhv_pois, agg_yellow_pois, agg_fhv_pois, n_days_yellow, n_days_fhv)

In [ ]:
result_yellow = result_yellow.toPandas()
result_fhv = result_fhv.toPandas()

In [ ]:
# result_yellow.write.mode("overwrite").parquet(OUTPUT_Y_POIS_DEMAND, compression = "snappy")
# result_fhv.write.mode("overwrite").parquet(OUTPUT_FHV_POIS_DEMAND, compression = "snappy")

## LOAD CLEAN DEMANDS

In [ ]:
df_demand_yellow = result_yellow
df_demand_fhv = result_fhv
df_pois_low = df_pois[['id', 'low_SES_density']] 
df_demand_yellow = pd.merge(df_demand_yellow, df_pois_low, how= "left", left_on = "PULocationID", right_on="id")
df_demand_fhv = pd.merge(df_demand_fhv, df_pois_low, how= "left", left_on = "PULocationID", right_on="id")

In [ ]:
OUTPUT_Y_POIS_DEMAND = projectRoot / "data" / "clean" /"wealthAnalysis" / "yellow_zone_demand_pois.parquet"
OUTPUT_FHV_POIS_DEMAND = projectRoot / "data" / "clean" /"wealthAnalysis" / "fhv_zone_demand_pois.parquet"
df_demand_yellow.to_parquet(OUTPUT_Y_POIS_DEMAND, compression = "snappy")
df_demand_fhv.to_parquet(OUTPUT_FHV_POIS_DEMAND, compression = "snappy")

In [ ]:
#spark.stop()

In [ ]:
df_demand_yellow.columns